# Phase 1 - Part 2: KNN & Naïve Bayes
**Medical Image Diagnosis System** | Cairo University - Supervised Learning, Spring 2026

Instructor: Dr. Ghada Dahy

In [4]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from skimage.feature import hog, local_binary_pattern
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, roc_curve, auc, confusion_matrix, classification_report
from sklearn.multiclass import OneVsRestClassifier
import os, warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries imported successfully!')

Libraries imported successfully!


In [2]:
# Config
CSV_PATH   = 'sample_labels.csv'
IMAGE_DIR  = 'sample/images/'
IMG_SIZE   = (128, 128)
MAX_IMAGES = 600
HOG_ORIENTATIONS    = 8
HOG_PIXELS_PER_CELL = (16, 16)
HOG_CELLS_PER_BLOCK = (1, 1)
LBP_P, LBP_R, LBP_METHOD = 8, 1, 'uniform'
CLASSES = ['No Finding', 'Infiltration', 'Effusion', 'Atelectasis', 'Nodule', 'Pneumonia']

In [3]:
# Load & Filter Dataset
df = pd.read_csv(CSV_PATH)
df = df[~df['Finding Labels'].str.contains('\|')]
df = df[df['Finding Labels'].isin(CLASSES)]
df = df.groupby('Finding Labels').apply(lambda x: x.sample(min(len(x), MAX_IMAGES), random_state=42)).reset_index(drop=True)
print(f'Dataset size: {len(df)}')
print(df['Finding Labels'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'sample_labels.csv'

In [5]:
# Feature Extraction: HOG + LBP
def extract_features(img_path):
    img = Image.open(img_path).convert('L').resize(IMG_SIZE)
    arr = np.array(img)
    hog_feat = hog(arr, orientations=HOG_ORIENTATIONS,
                   pixels_per_cell=HOG_PIXELS_PER_CELL,
                   cells_per_block=HOG_CELLS_PER_BLOCK, feature_vector=True)
    lbp = local_binary_pattern(arr, HOG_ORIENTATIONS, LBP_R, method=LBP_METHOD)
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=HOG_ORIENTATIONS+2,
                                range=(0, HOG_ORIENTATIONS+2), density=True)
    return np.concatenate([hog_feat, lbp_hist])

features, labels = [], []
for _, row in df.iterrows():
    path = os.path.join(IMAGE_DIR, row['Image Index'])
    if os.path.exists(path):
        features.append(extract_features(path))
        labels.append(row['Finding Labels'])

X = np.array(features)
le = LabelEncoder()
y = le.fit_transform(labels)
print(f'Features shape: {X.shape} | Classes: {le.classes_}')

NameError: name 'df' is not defined

In [ ]:
# Train/Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [6]:
# KNN Classifier
knn_grid = GridSearchCV(KNeighborsClassifier(), {'n_neighbors': [3,5,7,9]}, cv=3, scoring='f1_macro', n_jobs=-1)
knn_grid.fit(X_train, y_train)
best_knn  = knn_grid.best_estimator_
y_pred_knn = best_knn.predict(X_test)
knn_acc = accuracy_score(y_test, y_pred_knn)
knn_f1m = f1_score(y_test, y_pred_knn, average='macro')
knn_f1w = f1_score(y_test, y_pred_knn, average='weighted')
print(f'Best k={knn_grid.best_params_["n_neighbors"]} | Accuracy={knn_acc:.4f} | F1-macro={knn_f1m:.4f} | F1-weighted={knn_f1w:.4f}')

NameError: name 'X_train' is not defined

In [ ]:
# Naive Bayes Classifier
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
nb_acc = accuracy_score(y_test, y_pred_nb)
nb_f1m = f1_score(y_test, y_pred_nb, average='macro')
nb_f1w = f1_score(y_test, y_pred_nb, average='weighted')
print(f'Accuracy={nb_acc:.4f} | F1-macro={nb_f1m:.4f} | F1-weighted={nb_f1w:.4f}')

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, y_pred, title in zip(axes, [y_pred_knn, y_pred_nb], ['KNN', 'Naive Bayes']):
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
    ax.set_title(f'{title} - Confusion Matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

In [ ]:
# ROC Curve
n_classes   = len(le.classes_)
y_test_bin  = label_binarize(y_test, classes=range(n_classes))
fig, axes   = plt.subplots(1, 2, figsize=(14, 5))
for ax, base_model, title in zip(
        axes,
        [KNeighborsClassifier(**knn_grid.best_params_), GaussianNB()],
        ['KNN', 'Naive Bayes']):
    model = OneVsRestClassifier(base_model)
    model.fit(X_train, y_train)
    y_score = model.predict_proba(X_test)
    for i, cls in enumerate(le.classes_):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        ax.plot(fpr, tpr, label=f'{cls} (AUC={auc(fpr,tpr):.2f})')
    ax.plot([0,1],[0,1],'k--')
    ax.set_title(f'{title} - ROC Curve')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Classification Report
print('=== KNN ===')
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))
print('=== Naive Bayes ===')
print(classification_report(y_test, y_pred_nb,  target_names=le.classes_))